# MorphoGraph-MOF — revised full atomistic pipeline

This notebook accepts an ordered MOF CIF and exports the real periodic framework, generated surface slabs, surface-pair energy diagnostics, atomistic particles clipped from the MOF structure, and a result archive.

Important scientific safeguards:

- It does not use the asymmetric-slab average `(E_slab - N E_bulk)/(2A)` as though it were one face's surface energy. For asymmetric slabs, it stores `(E_slab - N E_bulk)/A = gamma_top + gamma_bottom` and attempts to separate the terms only when the linear system is full rank.
- Overdetermined pair fits must pass your stated RMS residual tolerance. A full-rank square system is marked as algebraically identified but not independently checked by redundant equations.
- A CHGNet Wulff morphology is created only when all required signed facet orientations have positive, identifiable energies. Missing values are never filled with guessed or equal energies.
- BFDH-like and equal-gamma morphologies are labelled geometry controls, not CHGNet predictions.
- The clipped atomistic particles are visualization geometries, not passivated or separately relaxed nanoparticles.

The source has been syntax-checked and its half-space polyhedron helper has been smoke-tested. A full CHGNet run still depends on the CIF, runtime package versions, and whether the slab-energy equations are identifiable for that MOF.

In [ ]:
# Run this cell first. Wait for installation to finish before running the pipeline.
%pip install pymatgen chgnet py3Dmol scipy pandas ipython

# Verify the critical packages before starting the calculation.
import pymatgen, chgnet, scipy, pandas
print("Dependency check passed")
print("pymatgen:", getattr(pymatgen, "__version__", "version unavailable"))
print("CHGNet:", getattr(chgnet, "__version__", "version unavailable"))
print("SciPy:", scipy.__version__)
print("pandas:", pandas.__version__)


## Run the full pipeline

After the install/check cell finishes, run the next cell. It prompts for each scientific and visualization parameter. Press Enter to use the displayed default, or enter a documented value of your own. On first run, upload the MOF CIF when prompted.

Use an ordered and chemically reviewed CIF. CIFs with partial occupancies are rejected rather than silently converted into a guessed ordered structure.

In [ ]:
# MorphoGraph-MOF — revised, diagnostic, atomistic surface/morphology pipeline
# Intended for Google Colab. Run the install cell from the companion notebook first.
#
# Scientific design:
# 1) Use the primitive-standard cell where possible to avoid unnecessary supercells.
# 2) Relax the bulk at fixed cell and use its energy per atom as the reference.
# 3) Generate directed Miller-orientation representatives using the actual point group.
# 4) Relax stoichiometric slabs, including asymmetric slabs.
# 5) For each facet family, fit E_slab - N E_bulk = A*(gamma_top + gamma_bottom).
#    Individual termination energies are accepted only if the equations are full-rank,
#    the fit residual meets the user-entered tolerance, and fitted gammas are positive.
# 6) Construct a signed-normal Wulff polyhedron only if every requested directed facet
#    class has a validated surface-energy estimate. No missing facet energies are invented.
# 7) Also construct clearly labelled geometry-only controls (equal-gamma and BFDH-like).
# 8) Clip repeated atoms from the actual periodic CIF inside each polyhedron for visualization.
#
# Important: this is a computational workflow, not a guarantee that CHGNet is accurate for
# every MOF. Check convergence, slab thickness/vacuum sensitivity, and DFT validation before
# scientific publication. Disordered CIFs are rejected rather than silently altered.

import os
import re
import json
import math
import hashlib
import shutil
from datetime import datetime
import inspect
import itertools
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import redirect_stdout, redirect_stderr
from io import StringIO
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from pymatgen.core import Structure, Lattice, Molecule
from pymatgen.core.surface import SlabGenerator
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.local_env import CrystalNN
from chgnet.model import CHGNet
from chgnet.model.dynamics import StructOptimizer
from scipy.spatial import ConvexHull, HalfspaceIntersection, QhullError

warnings.filterwarnings("once", category=DeprecationWarning)

# -----------------------------------------------------------------------------
# 1. User-visible parameters. Enter accepts the shown default.
# -----------------------------------------------------------------------------
def ask_int(prompt, default, minimum=1):
    while True:
        raw = input(f"{prompt} [{default}]: ").strip()
        try:
            value = int(raw) if raw else int(default)
            if value < minimum:
                raise ValueError(f"must be >= {minimum}")
            return value
        except Exception as exc:
            print(f"Invalid value ({exc}). Try again.")


def ask_float(prompt, default, minimum=0.0):
    while True:
        raw = input(f"{prompt} [{default}]: ").strip()
        try:
            value = float(raw) if raw else float(default)
            if not np.isfinite(value) or value <= minimum:
                raise ValueError(f"must be finite and > {minimum}")
            return value
        except Exception as exc:
            print(f"Invalid value ({exc}). Try again.")


MAX_MILLER_INDEX = ask_int("Maximum absolute Miller index", 2, 1)
SLAB_THICKNESS_A = ask_float("Minimum slab thickness (angstrom)", 15.0, 0.0)
VACUUM_A = ask_float("Vacuum thickness (angstrom)", 20.0, 0.0)
FORCE_TOL_EV_A = ask_float("Maximum force tolerance (eV/angstrom)", 0.05, 0.0)
MAX_RELAX_STEPS = ask_int("Maximum relaxation steps per structure", 500, 1)
PARTICLE_RADIUS_A = ask_float("Displayed particle maximum radius (angstrom; visualization only)", 15.0, 0.0)
DISPLAY_REPEATS = ask_int("Bulk network repeats per axis", 2, 1)
SYMPREC_A = ask_float("Symmetry tolerance (angstrom)", 0.1, 0.0)
SURFACE_FINGERPRINT_DEPTH_A = ask_float("Surface fingerprint depth from each face (angstrom)", 4.0, 0.0)
FINGERPRINT_DISTANCE_TOL_A = ask_float("Surface fingerprint distance quantization (angstrom)", 0.1, 0.0)
PAIR_FIT_RMS_TOL_EV_A2 = ask_float("Maximum pair-energy fit RMS residual (eV/angstrom^2)", 0.02, 0.0)
TERMINATION_SHIFT_TOL_FRACTIONAL = ask_float(
    "Slab termination-shift deduplication tolerance (fractional; pymatgen ftol)", 0.1, 0.0
)

# -----------------------------------------------------------------------------
# 2. Load CIF and obtain a primitive-standard working cell if possible.
# Miller indices in this run are referenced to the recorded working cell.
# -----------------------------------------------------------------------------
try:
    CIF_PATH
except NameError:
    try:
        from google.colab import files
        uploaded = files.upload()
        cif_names = [name for name in uploaded if name.lower().endswith(".cif")]
        if not cif_names:
            raise ValueError("No .cif file was uploaded. Upload an ordered MOF CIF.")
        CIF_PATH = cif_names[0]
    except ImportError:
        CIF_PATH = input("Path to ordered MOF CIF: ").strip()

CIF_PATH = str(CIF_PATH)
if not os.path.isfile(CIF_PATH):
    raise FileNotFoundError(f"CIF not found: {CIF_PATH}")

input_structure = Structure.from_file(CIF_PATH)
if len(input_structure) == 0:
    raise ValueError("The CIF parser returned an empty structure.")
if not input_structure.is_ordered:
    raise ValueError(
        "This CIF has partial/disordered occupancies. Make and validate an ordered atomic model first; "
        "the pipeline will not silently choose one atom from a mixed-occupancy site."
    )

standardization_notes = []
try:
    _sga = SpacegroupAnalyzer(input_structure, symprec=SYMPREC_A)
    _primitive = _sga.get_primitive_standard_structure()
    if _primitive is not None and len(_primitive) > 0 and _primitive.is_ordered:
        working_structure = _primitive
        standardization_notes.append("Using primitive-standard structure to reduce redundant atoms in slab calculations.")
    else:
        working_structure = input_structure.copy()
        standardization_notes.append("Primitive standardization returned no suitable structure; using input CIF cell.")
except Exception as exc:
    working_structure = input_structure.copy()
    standardization_notes.append(f"Primitive standardization failed; using input CIF cell ({type(exc).__name__}: {exc}).")

formula = working_structure.composition.reduced_formula
safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(CIF_PATH).stem).strip("_") or "MOF"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_DIR = Path(f"MorphoGraph_MOF_results_{safe_name}_{RUN_ID}")
INITIAL_SLAB_DIR = RESULTS_DIR / "slabs_initial_unique"
RELAXED_SLAB_DIR = RESULTS_DIR / "slabs_relaxed"
PARTICLE_DIR = RESULTS_DIR / "particles"
for _dir in (RESULTS_DIR, INITIAL_SLAB_DIR, RELAXED_SLAB_DIR, PARTICLE_DIR):
    _dir.mkdir(parents=True, exist_ok=True)
working_structure.to(filename=str(RESULTS_DIR / f"{safe_name}_working_cell.cif"))
print("\n================ MOF INPUT ================")
print(f"Input file       : {CIF_PATH}")
print(f"Formula          : {formula}")
print(f"Input atoms      : {len(input_structure)}")
print(f"Working-cell atoms: {len(working_structure)}")
print(f"Lattice lengths (A): {np.round(working_structure.lattice.abc, 5)}")
print(f"Lattice angles (deg): {np.round(working_structure.lattice.angles, 4)}")
print(f"Cell volume (A^3): {working_structure.volume:.5f}")
for note in standardization_notes:
    print("Note:", note)
print("Miller indices will be referenced to the working-cell setting written to the output CIF.")
print("===========================================\n")

# -----------------------------------------------------------------------------
# 3. General utilities: formulas, energies, Miller orbits, and slab fingerprints.
# -----------------------------------------------------------------------------
def normalize_hkl(hkl):
    vals = tuple(int(round(v)) for v in hkl)
    if vals == (0, 0, 0):
        return None
    g = math.gcd(math.gcd(abs(vals[0]), abs(vals[1])), abs(vals[2]))
    if g > 1:
        vals = tuple(v // g for v in vals)
    return vals


def enumerate_primitive_directed_hkls(max_index):
    values = set()
    for raw in itertools.product(range(-max_index, max_index + 1), repeat=3):
        hkl = normalize_hkl(raw)
        if hkl is not None and max(abs(v) for v in hkl) <= max_index:
            values.add(hkl)
    return sorted(values)


def directed_miller_orbits(structure, max_index, symprec):
    """Reduce signed Miller indices only by actual point-group operations (not by ± equivalence)."""
    all_hkls = set(enumerate_primitive_directed_hkls(max_index))
    notes = []
    reciprocal_ops = []
    try:
        analyzer = SpacegroupAnalyzer(structure, symprec=symprec)
        operations = analyzer.get_point_group_operations(cartesian=False)
        for op in operations:
            rot = np.asarray(op.rotation_matrix, dtype=float)
            transform = np.linalg.inv(rot).T
            if not np.allclose(transform, np.rint(transform), atol=1e-5):
                notes.append("Skipped a non-integer reciprocal rotation in this cell setting; it was not used to merge facets.")
                continue
            reciprocal_ops.append(np.rint(transform).astype(int))
    except Exception as exc:
        notes.append(f"Point-group reduction failed ({type(exc).__name__}: {exc}); use identity-only facet orbits.")
    if not reciprocal_ops:
        reciprocal_ops = [np.eye(3, dtype=int)]

    unseen = set(all_hkls)
    representatives = []
    orbit_map = {}
    direction_to_rep = {}
    while unseen:
        seed = min(unseen)
        orbit = set()
        frontier = [seed]
        while frontier:
            current = frontier.pop()
            if current in orbit:
                continue
            orbit.add(current)
            for transform in reciprocal_ops:
                mapped_arr = transform @ np.asarray(current, dtype=int)
                mapped = normalize_hkl(mapped_arr)
                if mapped is None or max(abs(v) for v in mapped) > max_index or mapped not in all_hkls:
                    continue
                if mapped not in orbit:
                    frontier.append(mapped)
        # Ensure an operation with numerical setting issues never removes unaccounted indices.
        orbit &= all_hkls
        if not orbit:
            orbit = {seed}
        rep = min(orbit)
        representatives.append(rep)
        orbit_map[rep] = sorted(orbit)
        for direction in orbit:
            direction_to_rep[direction] = rep
        unseen -= orbit
    representatives = sorted(set(representatives))
    return representatives, orbit_map, direction_to_rep, sorted(all_hkls), notes


def element_amounts(structure):
    return {str(k): float(v) for k, v in structure.composition.get_el_amt_dict().items()}


def composition_is_proportional(candidate, bulk, rtol=1e-5, atol=1e-5):
    a = element_amounts(candidate)
    b = element_amounts(bulk)
    if not b or set(a) != set(b):
        return False
    ratios = [a[e] / b[e] for e in b]
    return bool(ratios) and all(np.isclose(r, ratios[0], rtol=rtol, atol=atol) for r in ratios)


def surface_area_A2(structure):
    matrix = np.asarray(structure.lattice.matrix, dtype=float)
    # SlabGenerator reorients the lattice so rows 0 and 1 span the surface plane.
    area = float(np.linalg.norm(np.cross(matrix[0], matrix[1])))
    if not np.isfinite(area) or area <= 0:
        raise ValueError(f"Invalid in-plane slab area: {area}")
    return area


def to_numpy(value):
    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=float)


def model_energy_force(model, structure):
    """Return total energy (eV), energy/atom (eV/atom), and maximum force (eV/A)."""
    pred = model.predict_structure(structure, task="ef")
    e = to_numpy(pred["e"]).reshape(-1)
    if not len(e) or not np.isfinite(e[0]):
        raise ValueError("CHGNet returned a non-finite energy.")
    forces = to_numpy(pred["f"])
    if forces.shape != (len(structure), 3) or not np.isfinite(forces).all():
        raise ValueError(f"Unexpected or non-finite force array: {forces.shape}")
    epa = float(e[0])
    fmax = float(np.max(np.linalg.norm(forces, axis=1))) if len(forces) else 0.0
    return epa * len(structure), epa, fmax


def relax_fixed_cell(optimizer, structure, fmax, steps):
    """Version-aware fixed-cell relaxation. Raise real optimizer errors rather than hiding them."""
    parameters = inspect.signature(optimizer.relax).parameters
    accepts_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in parameters.values())
    kwargs = {"fmax": fmax, "steps": steps}
    if "relax_cell" in parameters or accepts_kwargs:
        kwargs["relax_cell"] = False
    if "verbose" in parameters or accepts_kwargs:
        kwargs["verbose"] = False
    captured = StringIO()
    with redirect_stdout(captured), redirect_stderr(captured):
        result = optimizer.relax(structure.copy(), **kwargs)
    if not isinstance(result, dict) or result.get("final_structure") is None:
        raise RuntimeError("CHGNet optimizer did not return a final_structure.")
    final_structure = result["final_structure"]
    if not isinstance(final_structure, Structure) or len(final_structure) != len(structure):
        raise RuntimeError("CHGNet returned an unexpected final structure or changed atom count.")
    return final_structure


def fmt_hkl(hkl):
    return "(" + ",".join(str(int(x)) for x in hkl) + ")"


def file_hkl(hkl):
    return "_".join((f"m{abs(x)}" if x < 0 else str(x)) for x in hkl)


def surface_fingerprint(slab, side, depth_A, distance_tol_A):
    """Translation/in-plane-periodic fingerprint of atoms close to one slab face.

    The top and bottom fingerprint is formed from atoms within a user-controlled depth.
    It uses species, distance-from-face and periodic pair distances; no atom identity labels
    or hand-picked MOF-specific indices are used.
    """
    matrix = np.asarray(slab.lattice.matrix, dtype=float)
    a_vec, b_vec, c_vec = matrix
    normal = np.cross(a_vec, b_vec)
    norm = float(np.linalg.norm(normal))
    if norm <= 0:
        raise ValueError("Cannot determine slab normal for surface fingerprint.")
    normal /= norm
    if np.dot(normal, c_vec) < 0:
        normal *= -1
    coords = np.asarray(slab.cart_coords, dtype=float)
    heights = coords @ normal
    edge = float(np.max(heights) if side == "top" else np.min(heights))
    depth = (edge - heights) if side == "top" else (heights - edge)
    indices = np.where(depth <= depth_A + 1e-8)[0]
    if len(indices) == 0:
        indices = np.array([int(np.argmax(heights) if side == "top" else np.argmin(heights))])
    symbols = [slab[i].specie.symbol for i in indices]
    frac = np.asarray(slab.frac_coords, dtype=float)

    atom_signature = tuple(sorted(
        (symbols[q], int(round(float(depth[i]) / distance_tol_A)))
        for q, i in enumerate(indices)
    ))
    pair_counts = Counter()
    inplane_shifts = list(itertools.product((-1, 0, 1), repeat=2))
    for q in range(len(indices)):
        i = int(indices[q])
        for r in range(q + 1, len(indices)):
            j = int(indices[r])
            delta_xy = frac[i, :2] - frac[j, :2]
            best_parallel = np.inf
            for shift in inplane_shifts:
                vec_xy = (delta_xy + np.asarray(shift, dtype=float)) @ matrix[:2]
                dxy = float(np.linalg.norm(vec_xy))
                if dxy < best_parallel:
                    best_parallel = dxy
            dz = float(depth[i] - depth[j])
            dist = math.sqrt(best_parallel * best_parallel + dz * dz)
            sp_pair = tuple(sorted((slab[i].specie.symbol, slab[j].specie.symbol)))
            quantized = int(round(dist / distance_tol_A))
            pair_counts[(sp_pair, quantized)] += 1
    canonical = (atom_signature, tuple(sorted(pair_counts.items(), key=repr)))
    return hashlib.sha1(repr(canonical).encode("utf-8")).hexdigest()[:20]

# -----------------------------------------------------------------------------
# 4. Relax the bulk reference at fixed cell.
# -----------------------------------------------------------------------------
model = None
optimizer = None
bulk_final = working_structure.copy()
bulk_total_energy_eV = None
bulk_energy_per_atom_eV = None
bulk_fmax_eV_A = None
bulk_ok = False
bulk_error = None

try:
    model = CHGNet.load()
    optimizer = StructOptimizer(model=model)
    print("CHGNet loaded. Relaxing the bulk at fixed cell...")
    bulk_final = relax_fixed_cell(optimizer, working_structure, FORCE_TOL_EV_A, MAX_RELAX_STEPS)
    bulk_total_energy_eV, bulk_energy_per_atom_eV, bulk_fmax_eV_A = model_energy_force(model, bulk_final)
    bulk_ok = bool(bulk_fmax_eV_A <= FORCE_TOL_EV_A)
    bulk_final.to(filename=str(RESULTS_DIR / f"{safe_name}_bulk_relaxed_fixed_cell.cif"))
    print(f"Bulk total energy : {bulk_total_energy_eV:.8f} eV")
    print(f"Bulk energy/atom  : {bulk_energy_per_atom_eV:.8f} eV/atom")
    print(f"Bulk max force    : {bulk_fmax_eV_A:.6f} eV/A")
    print(f"Bulk converged    : {bulk_ok}")
    if not bulk_ok:
        bulk_error = "Bulk max force exceeded the selected tolerance; surface energies are disabled."
except Exception as exc:
    bulk_error = f"{type(exc).__name__}: {exc}"
    print("FATAL FOR MLIP SURFACE ENERGIES: bulk relaxation/reference failed:", bulk_error)
    model = None
    optimizer = None
    bulk_ok = False

# -----------------------------------------------------------------------------
# 5. Generate directed (signed) Miller orientation orbits.
# -----------------------------------------------------------------------------
miller_reps, miller_orbits, direction_to_rep, all_directed_hkls, miller_notes = directed_miller_orbits(
    working_structure, MAX_MILLER_INDEX, SYMPREC_A
)
print(f"\nRequested signed Miller range: |h|,|k|,|l| <= {MAX_MILLER_INDEX}")
print(f"Directed primitive Miller orientations: {len(all_directed_hkls)}")
print(f"Symmetry-reduced directed surface classes: {len(miller_reps)}")
for note in miller_notes:
    print("Miller note:", note)
print("No signed orientation is merged with its opposite unless a real point-group operation maps them.")

# -----------------------------------------------------------------------------
# 6. Generate stoichiometric slabs; relax asymmetric slabs too. Energies are pair sums.
# -----------------------------------------------------------------------------
records = []
unique_relaxed_records = []
primary_pair_files = {}

for facet_i, hkl in enumerate(miller_reps, start=1):
    print(f"\n[{facet_i}/{len(miller_reps)}] Directed facet class representative {fmt_hkl(hkl)}")
    generation_errors = []
    generated = None
    generator_primitive_mode = None
    for use_primitive_surface_cell in (True, False):
        try:
            generator = SlabGenerator(
                initial_structure=bulk_final,
                miller_index=hkl,
                min_slab_size=SLAB_THICKNESS_A,
                min_vacuum_size=VACUUM_A,
                in_unit_planes=False,
                center_slab=True,
                primitive=use_primitive_surface_cell,
                reorient_lattice=True,
            )
            get_params = inspect.signature(generator.get_slabs).parameters
            get_kwargs = {}
            if "symmetrize" in get_params:
                get_kwargs["symmetrize"] = False  # asymmetric faces are handled by pair-energy equations
            if "filter_out_sym_slabs" in get_params:
                get_kwargs["filter_out_sym_slabs"] = False  # retain all unique termination candidates
            if "ftol" in get_params:
                get_kwargs["ftol"] = TERMINATION_SHIFT_TOL_FRACTIONAL
            generated = generator.get_slabs(**get_kwargs)
            generator_primitive_mode = bool(use_primitive_surface_cell)
            break
        except Exception as exc:
            generation_errors.append(
                f"primitive={use_primitive_surface_cell}: {type(exc).__name__}: {exc}"
            )
            generated = None

    if generated is None:
        exc_detail = " | ".join(generation_errors)
        records.append({
            "h": hkl[0], "k": hkl[1], "l": hkl[2], "miller": fmt_hkl(hkl),
            "termination": 0, "n_atoms": 0, "area_A2": np.nan,
            "stoichiometric": False, "symmetric_diagnostic": False,
            "top_fingerprint": "", "bottom_fingerprint": "",
            "slab_total_energy_eV": np.nan, "pair_excess_eV_A2": np.nan,
            "fit_rank": 0, "fit_unknowns": 0, "fit_rms_eV_A2": np.nan,
            "status": "slab_generation_failed", "detail": exc_detail,
            "initial_slab_file": "", "relaxed_slab_file": "",
            "slab_generator_primitive_surface_cell": None,
        })
        print(f"  Slab generation failed in both surface-cell modes: {exc_detail}")
        continue

    print(f"  Slab surface-cell mode: primitive={generator_primitive_mode}")
    print(f"  Generated raw termination candidates: {len(generated)}")
    seen_pairs = set()
    retained_this_facet = 0
    for term_i, slab in enumerate(generated, start=1):
        base = {
            "h": hkl[0], "k": hkl[1], "l": hkl[2], "miller": fmt_hkl(hkl),
            "termination": int(term_i), "n_atoms": int(len(slab)),
            "area_A2": np.nan, "stoichiometric": False,
            "symmetric_diagnostic": False, "symmetric_after_relaxation": False,
            "top_fingerprint": "", "bottom_fingerprint": "",
            "top_fingerprint_relaxed": "", "bottom_fingerprint_relaxed": "",
            "slab_total_energy_eV": np.nan, "pair_excess_eV_A2": np.nan,
            "fit_rank": 0, "fit_unknowns": 0, "fit_rms_eV_A2": np.nan,
            "status": "", "detail": "", "initial_slab_file": "", "relaxed_slab_file": "",
            "slab_generator_primitive_surface_cell": generator_primitive_mode,
        }
        try:
            area = surface_area_A2(slab)
            base["area_A2"] = area
            base["stoichiometric"] = bool(composition_is_proportional(slab, working_structure))
            try:
                base["symmetric_diagnostic"] = bool(slab.is_symmetric(symprec=SYMPREC_A))
            except Exception:
                base["symmetric_diagnostic"] = False
            if not base["stoichiometric"]:
                base["status"] = "excluded_nonstoichiometric"
                base["detail"] = "Non-stoichiometric slab: chemical-potential reference terms are required; excluded from this fit."
                records.append(base)
                continue
            top_fp = surface_fingerprint(slab, "top", SURFACE_FINGERPRINT_DEPTH_A, FINGERPRINT_DISTANCE_TOL_A)
            bottom_fp = surface_fingerprint(slab, "bottom", SURFACE_FINGERPRINT_DEPTH_A, FINGERPRINT_DISTANCE_TOL_A)
            base["top_fingerprint"] = top_fp
            base["bottom_fingerprint"] = bottom_fp
            pair_key = (int(len(slab)), round(area, 4), top_fp, bottom_fp)
            if pair_key in seen_pairs:
                base["status"] = "duplicate_surface_pair_not_relaxed"
                base["detail"] = "Same atom count, area, and ordered top/bottom surface fingerprints as an earlier candidate; skipped to avoid repeating the same slab relaxation."
                base["initial_slab_file"] = primary_pair_files.get((hkl, pair_key), "")
                records.append(base)
                continue
            seen_pairs.add(pair_key)
            retained_this_facet += 1
            initial_path = INITIAL_SLAB_DIR / f"{safe_name}_hkl_{file_hkl(hkl)}_term_{term_i:04d}.cif"
            slab.to(filename=str(initial_path))
            base["initial_slab_file"] = str(initial_path)
            primary_pair_files[(hkl, pair_key)] = str(initial_path)

            if model is None or optimizer is None or not bulk_ok or bulk_energy_per_atom_eV is None:
                base["status"] = "surface_energy_disabled"
                base["detail"] = bulk_error or "A converged bulk reference is unavailable."
                records.append(base)
                continue

            relaxed = relax_fixed_cell(optimizer, slab, FORCE_TOL_EV_A, MAX_RELAX_STEPS)
            slab_total_eV, _slab_epa, slab_fmax = model_energy_force(model, relaxed)
            base["fmax_eV_A"] = float(slab_fmax)
            if slab_fmax > FORCE_TOL_EV_A:
                base["status"] = "excluded_not_converged"
                base["detail"] = f"Relaxed slab max force {slab_fmax:.6f} eV/A exceeds tolerance {FORCE_TOL_EV_A:.6f} eV/A."
                records.append(base)
                continue
            if not composition_is_proportional(relaxed, working_structure):
                base["status"] = "excluded_relaxed_composition_mismatch"
                base["detail"] = "The relaxed slab composition no longer matches the bulk formula ratio."
                records.append(base)
                continue

            # Fingerprint the relaxed face terminations, because the final energy belongs to
            # the relaxed surface state. Large reconstructions become distinct fingerprint IDs.
            base["top_fingerprint_relaxed"] = surface_fingerprint(
                relaxed, "top", SURFACE_FINGERPRINT_DEPTH_A, FINGERPRINT_DISTANCE_TOL_A
            )
            base["bottom_fingerprint_relaxed"] = surface_fingerprint(
                relaxed, "bottom", SURFACE_FINGERPRINT_DEPTH_A, FINGERPRINT_DISTANCE_TOL_A
            )
            try:
                base["symmetric_after_relaxation"] = bool(relaxed.is_symmetric(symprec=SYMPREC_A))
            except Exception:
                base["symmetric_after_relaxation"] = False
            # If pymatgen verifies the relaxed slab's face-exchanging symmetry, the two
            # faces are the same surface state. Use one shared unknown in the pair equations
            # even if numerical fingerprinting separated them by a small geometric detail.
            if base["symmetric_after_relaxation"]:
                base["bottom_fingerprint_relaxed"] = base["top_fingerprint_relaxed"]
            # For possibly inequivalent faces this is gamma_top + gamma_bottom, NOT one gamma.
            pair_excess = (slab_total_eV - len(relaxed) * bulk_energy_per_atom_eV) / area
            relaxed_path = RELAXED_SLAB_DIR / f"{safe_name}_hkl_{file_hkl(hkl)}_term_{term_i:04d}.cif"
            relaxed.to(filename=str(relaxed_path))
            base["slab_total_energy_eV"] = float(slab_total_eV)
            base["pair_excess_eV_A2"] = float(pair_excess)
            base["fmax_eV_A"] = float(slab_fmax)
            base["relaxed_slab_file"] = str(relaxed_path)
            base["status"] = "pair_energy_available"
            base["detail"] = "Stoichiometric, force-converged slab. Stored value is the sum of both surface energies per area; individual gammas are solved below."
            records.append(base)
            unique_relaxed_records.append(base.copy())
            print(f"  Termination {term_i}: relaxed; N={len(relaxed)}; area={area:.3f} A^2; pair excess={pair_excess:.5f} eV/A^2; fmax={slab_fmax:.4f} eV/A")
        except Exception as exc:
            base["status"] = "slab_processing_failed"
            base["detail"] = f"{type(exc).__name__}: {exc}"
            records.append(base)
            print(f"  Termination {term_i} failed: {type(exc).__name__}: {exc}")
    print(f"  Unique stoichiometric surface pairs kept for relaxation: {retained_this_facet}")

# -----------------------------------------------------------------------------
# 7. Deconvolve pair energies into termination-specific surface energies.
# Each row satisfies excess/A = gamma(top fingerprint) + gamma(bottom fingerprint).
# Non-stoichiometric records are never included in this simple-reference fit.
# -----------------------------------------------------------------------------
records_df = pd.DataFrame(records)
if records_df.empty:
    records_df = pd.DataFrame(columns=[
        "h", "k", "l", "miller", "termination", "n_atoms", "area_A2", "stoichiometric",
        "symmetric_diagnostic", "symmetric_after_relaxation", "top_fingerprint", "bottom_fingerprint",
        "top_fingerprint_relaxed", "bottom_fingerprint_relaxed", "slab_total_energy_eV",
        "pair_excess_eV_A2", "fmax_eV_A", "fit_rank", "fit_unknowns", "fit_rms_eV_A2",
        "status", "detail", "initial_slab_file", "relaxed_slab_file"
    ])

surface_gamma_rows = []
fit_summary_rows = []
valid_pair_df = records_df[records_df.get("status", pd.Series(dtype=str)) == "pair_energy_available"].copy()

for hkl in miller_reps:
    group = valid_pair_df[
        (pd.to_numeric(valid_pair_df["h"], errors="coerce") == hkl[0]) &
        (pd.to_numeric(valid_pair_df["k"], errors="coerce") == hkl[1]) &
        (pd.to_numeric(valid_pair_df["l"], errors="coerce") == hkl[2])
    ].copy()
    fit_row = {
        "h": hkl[0], "k": hkl[1], "l": hkl[2], "miller": fmt_hkl(hkl),
        "n_pair_equations": int(len(group)), "n_surface_terminations": 0,
        "matrix_rank": 0, "matrix_columns": 0, "fit_rms_eV_A2": np.nan,
        "fit_status": "no_valid_pair_energies", "detail": "No stoichiometric, force-converged slab pairs were available."
    }
    if group.empty:
        fit_summary_rows.append(fit_row)
        continue

    terminations = sorted(set(group["top_fingerprint_relaxed"].astype(str)) | set(group["bottom_fingerprint_relaxed"].astype(str)))
    col_index = {fp: i for i, fp in enumerate(terminations)}
    A = np.zeros((len(group), len(terminations)), dtype=float)
    y = pd.to_numeric(group["pair_excess_eV_A2"], errors="coerce").to_numpy(dtype=float)
    for row_i, row in enumerate(group.itertuples()):
        A[row_i, col_index[str(row.top_fingerprint_relaxed)]] += 1.0
        A[row_i, col_index[str(row.bottom_fingerprint_relaxed)]] += 1.0
    rank = int(np.linalg.matrix_rank(A))
    n_equations = int(len(group))
    n_unknowns = int(len(terminations))
    n_redundant_equations = max(0, n_equations - n_unknowns)
    fit_row.update({
        "n_surface_terminations": n_unknowns,
        "matrix_rank": rank,
        "matrix_columns": n_unknowns,
        "n_redundant_equations": n_redundant_equations,
        "has_redundant_equations": bool(n_redundant_equations > 0),
        "fit_validation": "pending",
    })
    if rank < len(terminations):
        fit_row["fit_status"] = "underdetermined_surface_energies"
        fit_row["detail"] = (
            f"Pair-equation matrix rank {rank} is smaller than {len(terminations)} unknown termination energies. "
            "Individual surface energies cannot be uniquely separated from these slabs."
        )
        records_df.loc[(records_df["h"] == hkl[0]) & (records_df["k"] == hkl[1]) & (records_df["l"] == hkl[2]) & (records_df["status"] == "pair_energy_available"), "status"] = "pair_fit_underdetermined"
        records_df.loc[(records_df["h"] == hkl[0]) & (records_df["k"] == hkl[1]) & (records_df["l"] == hkl[2]) & (records_df["status"] == "pair_fit_underdetermined"), "detail"] = fit_row["detail"]
        fit_summary_rows.append(fit_row)
        continue

    solution, _resid, _rank, _singular = np.linalg.lstsq(A, y, rcond=None)
    residuals = A @ solution - y
    rms = float(np.sqrt(np.mean(residuals ** 2)))
    fit_row["fit_rms_eV_A2"] = rms
    if not np.isfinite(rms) or rms > PAIR_FIT_RMS_TOL_EV_A2:
        fit_row["fit_status"] = "fit_residual_exceeds_tolerance"
        fit_row["detail"] = (
            f"Pair-energy RMS residual {rms:.6f} eV/A^2 exceeds user tolerance "
            f"{PAIR_FIT_RMS_TOL_EV_A2:.6f} eV/A^2. Increase slab thickness or review surface fingerprints/model consistency."
        )
        fit_summary_rows.append(fit_row)
        records_df.loc[(records_df["h"] == hkl[0]) & (records_df["k"] == hkl[1]) & (records_df["l"] == hkl[2]) & (records_df["status"] == "pair_energy_available"), "status"] = "pair_fit_residual_too_high"
        records_df.loc[(records_df["h"] == hkl[0]) & (records_df["k"] == hkl[1]) & (records_df["l"] == hkl[2]) & (records_df["status"] == "pair_fit_residual_too_high"), "detail"] = fit_row["detail"]
        continue

    fit_row["fit_status"] = "fit_ok"
    if n_redundant_equations > 0:
        fit_row["fit_validation"] = "overdetermined_residual_checked"
        fit_row["detail"] = (
            "Full-rank pair-equation solution with redundant equations; RMS residual is within the user-specified tolerance. "
            "This is an internal consistency check, not a substitute for slab-thickness/vacuum convergence or DFT validation."
        )
    else:
        fit_row["fit_validation"] = "full_rank_no_redundancy"
        fit_row["detail"] = (
            "The pair-equation matrix is full rank, so the termination energies are algebraically identifiable, "
            "but there are no redundant equations to independently test pair-model consistency. Treat as provisional "
            "until slab-thickness/vacuum convergence and independent validation are completed."
        )
    gamma_map = {fp: float(solution[i]) for fp, i in col_index.items()}
    for fp, gamma_eV_A2 in gamma_map.items():
        positive = bool(np.isfinite(gamma_eV_A2) and gamma_eV_A2 > 0)
        surface_gamma_rows.append({
            "h": hkl[0], "k": hkl[1], "l": hkl[2], "miller": fmt_hkl(hkl),
            "termination_fingerprint": fp, "gamma_eV_A2": gamma_eV_A2,
            "gamma_J_m2": gamma_eV_A2 * 16.02176634,
            "positive_gamma": positive, "fit_rank": rank, "fit_unknowns": len(terminations),
            "fit_rms_eV_A2": rms,
            "fit_status": "fit_ok" if positive else "fit_ok_nonpositive_gamma",
            "fit_validation": fit_row["fit_validation"],
            "n_pair_equations": n_equations,
            "n_redundant_equations": n_redundant_equations,
        })
    records_df.loc[(records_df["h"] == hkl[0]) & (records_df["k"] == hkl[1]) & (records_df["l"] == hkl[2]) & (records_df["status"] == "pair_energy_available"), "fit_rank"] = rank
    records_df.loc[(records_df["h"] == hkl[0]) & (records_df["k"] == hkl[1]) & (records_df["l"] == hkl[2]) & (records_df["status"] == "pair_energy_available"), "fit_unknowns"] = len(terminations)
    records_df.loc[(records_df["h"] == hkl[0]) & (records_df["k"] == hkl[1]) & (records_df["l"] == hkl[2]) & (records_df["status"] == "pair_energy_available"), "fit_rms_eV_A2"] = rms
    fit_summary_rows.append(fit_row)

fit_summary_df = pd.DataFrame(fit_summary_rows).drop_duplicates(subset=["h", "k", "l"], keep="last")
surface_gamma_df = pd.DataFrame(surface_gamma_rows)
fit_csv = RESULTS_DIR / f"{safe_name}_surface_pair_fit_summary.csv"
surface_gamma_csv = RESULTS_DIR / f"{safe_name}_surface_energies_individual.csv"
records_csv = RESULTS_DIR / f"{safe_name}_surface_pair_records.csv"
records_df.to_csv(records_csv, index=False)
fit_summary_df.to_csv(fit_csv, index=False)
surface_gamma_df.to_csv(surface_gamma_csv, index=False)

print("\n================ SURFACE PAIR FIT SUMMARY ================")
if len(fit_summary_df):
    display(fit_summary_df)
else:
    print("No facet-fit diagnostics were produced.")
print("\nIndividual termination surface energies:")
if len(surface_gamma_df):
    display(surface_gamma_df)
else:
    print("No individual surface energies were identified from the pair equations.")
print(f"Pair records CSV : {records_csv}")
print(f"Fit summary CSV  : {fit_csv}")
print(f"Surface gamma CSV: {surface_gamma_csv}")

# -----------------------------------------------------------------------------
# 8. Convert solved termination energies into signed-direction energy scenarios.
# The minimum is the equilibrium termination choice; median/maximum are sensitivity
# scenarios only, not separate equilibrium predictions.
# -----------------------------------------------------------------------------
fit_lookup = {}
for row in surface_gamma_df.itertuples():
    fit_lookup.setdefault((int(row.h), int(row.k), int(row.l)), {})[str(row.termination_fingerprint)] = float(row.gamma_J_m2)

scenario_names = ("minimum", "median", "maximum")
raw_direction_scenarios = {name: defaultdict(list) for name in scenario_names}
raw_direction_invalid = defaultdict(list)
for rep in miller_reps:
    rep_group = valid_pair_df[
        (pd.to_numeric(valid_pair_df["h"], errors="coerce") == rep[0]) &
        (pd.to_numeric(valid_pair_df["k"], errors="coerce") == rep[1]) &
        (pd.to_numeric(valid_pair_df["l"], errors="coerce") == rep[2])
    ]
    fit_state = fit_summary_df[
        (fit_summary_df["h"] == rep[0]) & (fit_summary_df["k"] == rep[1]) & (fit_summary_df["l"] == rep[2])
    ]
    if fit_state.empty or fit_state.iloc[-1]["fit_status"] != "fit_ok":
        continue
    gamma_by_fp = fit_lookup.get(rep, {})
    top_fps = sorted(set(rep_group["top_fingerprint_relaxed"].astype(str)))
    bottom_fps = sorted(set(rep_group["bottom_fingerprint_relaxed"].astype(str)))

    def face_gamma_scenarios(fps, direction):
        if not fps:
            return None
        vals = [gamma_by_fp[fp] for fp in fps if fp in gamma_by_fp]
        if len(vals) != len(fps):
            return None
        if any(not np.isfinite(v) or v <= 0 for v in vals):
            raw_direction_invalid[direction].append(
                f"At least one fitted termination energy for {fmt_hkl(direction)} is non-positive or non-finite."
            )
            return None
        vals = np.asarray(vals, dtype=float)
        return {
            "minimum": float(np.min(vals)),
            "median": float(np.median(vals)),
            "maximum": float(np.max(vals)),
        }

    top_scenarios = face_gamma_scenarios(top_fps, rep)
    opposite = tuple(-v for v in rep)
    bottom_scenarios = face_gamma_scenarios(bottom_fps, opposite)
    if top_scenarios is not None:
        for direction in miller_orbits[rep]:
            for scenario in scenario_names:
                raw_direction_scenarios[scenario][direction].append(top_scenarios[scenario])
    if bottom_scenarios is not None:
        opposite_rep = direction_to_rep.get(opposite)
        if opposite_rep is not None:
            for direction in miller_orbits[opposite_rep]:
                for scenario in scenario_names:
                    raw_direction_scenarios[scenario][direction].append(bottom_scenarios[scenario])

# Multiple symmetry-equivalent paths may assign a direction more than once.
# Combine those assignments conservatively and retain diagnostics when they disagree.
direction_gamma_scenarios = {name: {} for name in scenario_names}
direction_gamma_notes = []
for scenario in scenario_names:
    for direction, vals in raw_direction_scenarios[scenario].items():
        vals = [float(v) for v in vals if np.isfinite(v) and v > 0]
        if not vals:
            continue
        if len(vals) > 1 and max(vals) - min(vals) > PAIR_FIT_RMS_TOL_EV_A2 * 16.02176634:
            direction_gamma_notes.append(
                f"Symmetry-equivalent estimates differ for {fmt_hkl(direction)} in the {scenario} scenario; combined by {scenario}."
            )
        if scenario == "minimum":
            combined = min(vals)
        elif scenario == "median":
            combined = float(np.median(vals))
        else:
            combined = max(vals)
        direction_gamma_scenarios[scenario][direction] = combined

direction_gamma_J_m2 = direction_gamma_scenarios["minimum"]

# -----------------------------------------------------------------------------
# 9. Signed-normal Wulff and explicitly labelled geometry-only controls.
# -----------------------------------------------------------------------------
def direction_normal(lattice, hkl):
    rec = np.asarray(lattice.reciprocal_lattice_crystallographic.matrix, dtype=float)
    normal = np.asarray(hkl, dtype=float) @ rec
    norm = float(np.linalg.norm(normal))
    if norm <= 0 or not np.isfinite(norm):
        raise ValueError(f"Invalid reciprocal normal for {hkl}.")
    return normal / norm


def build_polyhedron(label, directed_gamma, lattice):
    """Build an oriented Wulff polyhedron from half-spaces n.r <= gamma.

    Energies need only be relative: all distances are divided by the minimum positive gamma.
    This custom construction supports distinct energies for +hkl and -hkl.
    """
    usable = []
    for hkl, gamma in directed_gamma.items():
        hkl = normalize_hkl(hkl)
        gamma = float(gamma)
        if hkl is None or not np.isfinite(gamma) or gamma <= 0:
            continue
        normal = direction_normal(lattice, hkl)
        usable.append((hkl, normal, gamma))
    if len(usable) < 4:
        raise ValueError(f"Only {len(usable)} positive oriented facets are available; at least four are needed.")

    # Collapse exact/near-identical normals only, preserving the lowest face distance.
    unique = []
    for hkl, normal, gamma in usable:
        match = None
        for i, (_, old_normal, _) in enumerate(unique):
            if np.linalg.norm(normal - old_normal) < 1e-8:
                match = i
                break
        if match is None:
            unique.append((hkl, normal, gamma))
        elif gamma < unique[match][2]:
            unique[match] = (hkl, normal, gamma)
    scale = min(item[2] for item in unique)
    # scipy half-space convention: a.x + b <= 0, hence n.x - gamma/scale <= 0.
    halfspaces = np.asarray([np.r_[normal, -gamma / scale] for _, normal, gamma in unique], dtype=float)
    if not np.isfinite(halfspaces).all():
        raise ValueError("Non-finite Wulff halfspaces.")
    try:
        intersections = HalfspaceIntersection(halfspaces, np.zeros(3)).intersections
        # Deduplicate near-coincident vertices for a stable hull.
        vertices = []
        for point in intersections:
            if not any(np.linalg.norm(point - old) < 1e-6 for old in vertices):
                vertices.append(np.asarray(point, dtype=float))
        vertices = np.asarray(vertices, dtype=float)
        if vertices.ndim != 2 or vertices.shape[1] != 3 or len(vertices) < 4:
            raise ValueError("Half-space intersections did not produce a closed 3D polyhedron.")
        hull = ConvexHull(vertices)
        if not np.isfinite(hull.volume) or hull.volume <= 0:
            raise ValueError("Wulff polyhedron has no positive 3D volume.")
    except (QhullError, ValueError) as exc:
        raise ValueError(
            f"Could not create a closed bounded polyhedron from these orientations: {type(exc).__name__}: {exc}"
        ) from exc
    return {
        "label": label,
        "vertices": vertices,
        "simplices": np.asarray(hull.simplices, dtype=int),
        "equations": np.asarray(hull.equations, dtype=float),
        "volume_relative": float(hull.volume),
        "facet_count_input": int(len(unique)),
        "direction_gamma": {tuple(x[0]): float(x[2]) for x in unique},
    }


morphologies = []
scenario_labels = {
    "minimum": "CHGNet_minimum_termination_Wulff",
    "median": "CHGNet_median_termination_sensitivity_NOT_UNIQUE_EQUILIBRIUM",
    "maximum": "CHGNet_maximum_termination_sensitivity_NOT_UNIQUE_EQUILIBRIUM",
}
valid_scenario_shapes = {}
for scenario in scenario_names:
    gamma_map = direction_gamma_scenarios[scenario]
    missing_reps = [hkl for hkl in miller_reps if hkl not in gamma_map]
    if missing_reps:
        if scenario == "minimum":
            print("No complete CHGNet Wulff prediction: validated individual energies are missing for directed facet classes:")
            print(", ".join(fmt_hkl(h) for h in missing_reps))
            print("Missing energies are not replaced by arbitrary values. A partial-facet shape would be misleading.")
        continue
    try:
        item = build_polyhedron(scenario_labels[scenario], gamma_map, working_structure.lattice)
        if scenario == "minimum":
            morphologies.append(item)
            valid_scenario_shapes[scenario] = item
            print("Full-rank signed-orientation CHGNet Wulff polyhedron created.")
        if any(
            (not fit_summary_df[(fit_summary_df["h"] == rep[0]) & (fit_summary_df["k"] == rep[1]) & (fit_summary_df["l"] == rep[2])]["has_redundant_equations"].fillna(False).any())
            for rep in miller_reps
        ):
            print("Note: at least one facet fit has no redundant pair equations; its algebraic solution is identifiable but lacks an internal residual-based consistency check.")
        else:
            # Only keep sensitivity scenarios when some directional energy actually changes.
            base = direction_gamma_scenarios["minimum"]
            changed = any(
                not np.isclose(gamma_map.get(hkl, np.nan), base.get(hkl, np.nan), rtol=1e-5, atol=1e-8)
                for hkl in miller_reps
            )
            if changed:
                morphologies.append(item)
                valid_scenario_shapes[scenario] = item
                print(f"Created {scenario} termination-sensitivity shape (not a unique equilibrium prediction).")
    except Exception as exc:
        print(f"{scenario} CHGNet Wulff construction failed:", f"{type(exc).__name__}: {exc}")

if not direction_gamma_J_m2:
    print("No usable individual surface energies were identified; no CHGNet Wulff shape will be fabricated.")

# Geometry-only equal-gamma control, not a physical prediction.
equal_gamma = {tuple(hkl): 1.0 for hkl in all_directed_hkls}
try:
    morphologies.append(build_polyhedron("Equal_gamma_geometry_control_NOT_A_PREDICTION", equal_gamma, working_structure.lattice))
except Exception as exc:
    print("Equal-gamma geometry control failed:", f"{type(exc).__name__}: {exc}")

# BFDH-like d_hkl geometry baseline; not a surface-energy calculation.
bfdh_gamma = {}
for hkl in all_directed_hkls:
    try:
        d = float(working_structure.lattice.d_hkl(hkl))
        if np.isfinite(d) and d > 0:
            bfdh_gamma[tuple(hkl)] = d
    except Exception:
        continue
try:
    morphologies.append(build_polyhedron("BFDH_like_d_hkl_geometry_baseline_NOT_MLIP", bfdh_gamma, working_structure.lattice))
except Exception as exc:
    print("BFDH-like geometry baseline failed:", f"{type(exc).__name__}: {exc}")

# -----------------------------------------------------------------------------
# 10. Render the real framework and atomistically clip it into each candidate shape.
# -----------------------------------------------------------------------------
try:
    import py3Dmol
except Exception as exc:
    py3Dmol = None
    print("py3Dmol unavailable; files will still be exported:", exc)

bond_graph = None
connectivity_note = ""
try:
    bond_graph = CrystalNN().get_bonded_structure(bulk_final)
    connectivity_note = "Bonds are inferred by CrystalNN for display; they are not guaranteed to be CIF-authored bond topology."
except Exception as exc:
    connectivity_note = f"CrystalNN could not infer connectivity ({type(exc).__name__}: {exc}); atoms will be shown without bond cylinders."
    print("Connectivity warning:", connectivity_note)


def graph_edges_for_replicas(structure, graph, key_to_index, cell_ranges):
    if graph is None:
        return []
    edges = set()
    nsites = len(structure)
    for cell in itertools.product(*cell_ranges):
        cell = tuple(int(x) for x in cell)
        for i in range(nsites):
            skey = (i, cell)
            if skey not in key_to_index:
                continue
            try:
                connected_sites = graph.get_connected_sites(i)
            except Exception:
                continue
            for connected in connected_sites:
                j = int(connected.index)
                image = tuple(int(v) for v in connected.jimage)
                destination_cell = tuple(cell[q] + image[q] for q in range(3))
                dkey = (j, destination_cell)
                if dkey in key_to_index:
                    a, b = key_to_index[skey], key_to_index[dkey]
                    if a != b:
                        edges.add((min(a, b), max(a, b)))
    return sorted(edges)


def render_atoms(symbols, positions, edges=None, title="MOF", hull_vertices=None, hull_simplices=None):
    if py3Dmol is None or len(symbols) == 0:
        return None
    xyz_lines = [str(len(symbols)), title]
    xyz_lines.extend(f"{sym} {p[0]:.7f} {p[1]:.7f} {p[2]:.7f}" for sym, p in zip(symbols, positions))
    viewer = py3Dmol.view(width=900, height=650)
    viewer.addModel("\n".join(xyz_lines), "xyz")
    viewer.setStyle({}, {"sphere": {"scale": 0.27, "colorscheme": "Jmol"}})
    if edges:
        for i, j in edges:
            p1, p2 = positions[i], positions[j]
            try:
                viewer.addCylinder({
                    "start": {"x": float(p1[0]), "y": float(p1[1]), "z": float(p1[2])},
                    "end": {"x": float(p2[0]), "y": float(p2[1]), "z": float(p2[2])},
                    "radius": 0.065, "color": "#888888", "fromCap": 1, "toCap": 1,
                })
            except Exception as exc:
                print(f"Bond rendering stopped for {title}: {type(exc).__name__}: {exc}")
                break
    if hull_vertices is not None and hull_simplices is not None:
        hull_edges = set()
        for face in hull_simplices:
            for a, b in ((face[0], face[1]), (face[1], face[2]), (face[2], face[0])):
                hull_edges.add((min(int(a), int(b)), max(int(a), int(b))))
        for a, b in sorted(hull_edges):
            p1, p2 = hull_vertices[a], hull_vertices[b]
            try:
                viewer.addCylinder({
                    "start": {"x": float(p1[0]), "y": float(p1[1]), "z": float(p1[2])},
                    "end": {"x": float(p2[0]), "y": float(p2[1]), "z": float(p2[2])},
                    "radius": 0.04, "color": "#222222", "fromCap": 1, "toCap": 1,
                })
            except Exception as exc:
                print(f"Envelope rendering stopped for {title}: {type(exc).__name__}: {exc}")
                break
    viewer.setBackgroundColor("white")
    viewer.zoomTo()
    return viewer


def repeated_structure_view(structure, graph, repeats):
    repeat_tuple = (repeats, repeats, repeats) if isinstance(repeats, int) else tuple(int(x) for x in repeats)
    if len(repeat_tuple) != 3 or any(v < 1 for v in repeat_tuple):
        raise ValueError("repeats must be a positive integer or positive 3-tuple")
    ranges = [range(repeat_tuple[0]), range(repeat_tuple[1]), range(repeat_tuple[2])]
    matrix = np.asarray(structure.lattice.matrix, dtype=float)
    symbols, positions, key_to_index = [], [], {}
    for cell in itertools.product(*ranges):
        cell = tuple(int(x) for x in cell)
        shift = np.asarray(cell, dtype=float) @ matrix
        for i, site in enumerate(structure):
            key_to_index[(i, cell)] = len(positions)
            symbols.append(site.specie.symbol)
            positions.append(np.asarray(site.coords, dtype=float) + shift)
    positions = np.asarray(positions, dtype=float)
    positions -= positions.mean(axis=0)
    edges = graph_edges_for_replicas(structure, graph, key_to_index, ranges)
    return symbols, positions, edges


def make_atomistic_particle(morphology, structure, graph, radius_A):
    raw_vertices = np.asarray(morphology["vertices"], dtype=float)
    if raw_vertices.ndim != 2 or raw_vertices.shape[1] != 3 or len(raw_vertices) < 4:
        raise ValueError("Morphology has no valid 3D vertices")
    max_radius = float(np.max(np.linalg.norm(raw_vertices, axis=1)))
    if not np.isfinite(max_radius) or max_radius <= 0:
        raise ValueError("Invalid morphology scale")
    vertices = raw_vertices * (radius_A / max_radius)
    hull = ConvexHull(vertices)
    eq = hull.equations

    matrix = np.asarray(structure.lattice.matrix, dtype=float)
    inverse = np.linalg.inv(matrix)
    center = np.mean(np.asarray(structure.cart_coords, dtype=float), axis=0)
    base = np.asarray(structure.cart_coords, dtype=float) - center
    base_frac = base @ inverse
    vertex_frac = vertices @ inverse
    extent = np.ceil(np.max(np.abs(vertex_frac), axis=0) + np.max(np.abs(base_frac), axis=0)).astype(int) + 1
    ranges = [range(-int(n), int(n) + 1) for n in extent]

    symbols, xyz, key_to_index = [], [], {}
    for cell in itertools.product(*ranges):
        cell = tuple(int(x) for x in cell)
        shift = np.asarray(cell, dtype=float) @ matrix
        for i, site in enumerate(structure):
            pos = base[i] + shift
            if np.max(eq[:, :3] @ pos + eq[:, 3]) <= 1e-7:
                key = (i, cell)
                key_to_index[key] = len(xyz)
                symbols.append(site.specie.symbol)
                xyz.append(pos)
    if not xyz:
        raise ValueError("No atoms lie inside this polyhedron; increase the visualization radius or inspect the lattice setting")
    xyz = np.asarray(xyz, dtype=float)

    # Remove duplicate coordinates with a deterministic atom-index remapping.
    keep_indices = []
    seen = set()
    for i, (symbol, pos) in enumerate(zip(symbols, xyz)):
        key = (symbol, *np.round(pos, 6))
        if key not in seen:
            seen.add(key)
            keep_indices.append(i)
    if len(keep_indices) != len(xyz):
        old_to_new = {old: new for new, old in enumerate(keep_indices)}
        xyz = xyz[keep_indices]
        symbols = [symbols[i] for i in keep_indices]
        key_to_index = {key: old_to_new[idx] for key, idx in key_to_index.items() if idx in old_to_new}

    mins, maxs = xyz.min(axis=0), xyz.max(axis=0)
    lengths = np.maximum(maxs - mins + 10.0, 10.0)
    frac = (xyz - mins + 5.0) / lengths
    particle = Structure(
        Lattice.from_parameters(float(lengths[0]), float(lengths[1]), float(lengths[2]), 90, 90, 90),
        symbols, frac, coords_are_cartesian=False,
    )
    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", morphology["label"])
    cif_path = PARTICLE_DIR / f"{safe_name}_{stem}_atomistic_particle.cif"
    xyz_path = PARTICLE_DIR / f"{safe_name}_{stem}_atomistic_particle.xyz"
    particle.to(filename=str(cif_path))
    Molecule(symbols, xyz).to(filename=str(xyz_path))
    edges = graph_edges_for_replicas(structure, graph, key_to_index, ranges)
    return {
        "label": morphology["label"], "symbols": symbols, "xyz": xyz, "edges": edges,
        "vertices": vertices, "simplices": hull.simplices, "cif_path": str(cif_path),
        "xyz_path": str(xyz_path), "atom_count": len(symbols),
    }

morphology_rows = []
particle_objects = []
for morphology in morphologies:
    try:
        particle = make_atomistic_particle(morphology, bulk_final, bond_graph, PARTICLE_RADIUS_A)
        particle_objects.append(particle)
        methodology = (
            "CHGNet Wulff using minimum fitted termination gamma per signed orientation"
            if morphology["label"].startswith("CHGNet_minimum") else
            "Termination-energy sensitivity scenario; not a unique equilibrium prediction"
            if morphology["label"].startswith("CHGNet") else
            "BFDH-like d_hkl geometry baseline; not a surface-energy calculation"
            if morphology["label"].startswith("BFDH") else
            "Equal-gamma geometry control; not a physical morphology prediction"
        )
        morphology_rows.append({
            "morphology": morphology["label"], "method": methodology,
            "atom_count": particle["atom_count"], "display_radius_A": PARTICLE_RADIUS_A,
            "relative_polyhedron_volume": morphology["volume_relative"],
            "cif_file": particle["cif_path"], "xyz_file": particle["xyz_path"],
            "status": "ok", "detail": "Actual CIF-derived atoms clipped geometrically; not passivated or nanoparticle-relaxed."
        })
        print(f"Created {morphology['label']}: {particle['atom_count']} atoms")
    except Exception as exc:
        morphology_rows.append({
            "morphology": morphology.get("label", "unknown"), "method": "atomistic visualization",
            "atom_count": 0, "display_radius_A": PARTICLE_RADIUS_A,
            "relative_polyhedron_volume": morphology.get("volume_relative", np.nan),
            "cif_file": "", "xyz_file": "", "status": "particle_construction_failed",
            "detail": f"{type(exc).__name__}: {exc}"
        })
        print(f"Particle construction failed for {morphology.get('label')}: {type(exc).__name__}: {exc}")

morphology_df = pd.DataFrame(morphology_rows)
morphology_csv = RESULTS_DIR / f"{safe_name}_morphology_comparison.csv"
morphology_df.to_csv(morphology_csv, index=False)

# -----------------------------------------------------------------------------
# 11. Interactive outputs: actual network, representative slabs, and morphology gallery.
# -----------------------------------------------------------------------------
print("\n================ PERIODIC MOF NETWORK ================")
print(connectivity_note)
bulk_symbols, bulk_positions, bulk_edges = repeated_structure_view(bulk_final, bond_graph, DISPLAY_REPEATS)
bulk_viewer = render_atoms(
    bulk_symbols, bulk_positions, bulk_edges,
    title=f"{formula} — {DISPLAY_REPEATS}x{DISPLAY_REPEATS}x{DISPLAY_REPEATS} periodic MOF network",
)
if bulk_viewer is not None:
    display(bulk_viewer)

print("\n================ REPRESENTATIVE RELAXED SURFACE SLABS ================")
shown = set()
for row in records_df.itertuples():
    if not getattr(row, "relaxed_slab_file", "") or not isinstance(row.relaxed_slab_file, str):
        continue
    facet_key = (int(row.h), int(row.k), int(row.l))
    if facet_key in shown or not os.path.isfile(row.relaxed_slab_file):
        continue
    shown.add(facet_key)
    try:
        slab_struct = Structure.from_file(row.relaxed_slab_file)
        slab_graph = None
        try:
            slab_graph = CrystalNN().get_bonded_structure(slab_struct)
        except Exception:
            pass
        # Display a small lateral repeat only; slab CIF retains the full generated cell.
        syms, pos, edges = repeated_structure_view(slab_struct, slab_graph, (2, 2, 1))
        print(f"Facet class {row.miller}; termination {row.termination}; status {row.status}")
        viewer = render_atoms(syms, pos, edges, title=f"MOF slab {row.miller}, termination {row.termination}")
        if viewer is not None:
            display(viewer)
    except Exception as exc:
        print(f"Slab visualization failed for {getattr(row, 'miller', '?')}: {type(exc).__name__}: {exc}")

print("\n================ ATOMISTIC MORPHOLOGY GALLERY ================")
for particle in particle_objects:
    print(f"{particle['label']} — {particle['atom_count']} atoms")
    viewer = render_atoms(
        particle["symbols"], particle["xyz"], particle["edges"],
        title=particle["label"], hull_vertices=particle["vertices"], hull_simplices=particle["simplices"],
    )
    if viewer is not None:
        display(viewer)
if not particle_objects:
    print("No atomistic morphology particles were created. Check polyhedron and clipping diagnostics.")

# -----------------------------------------------------------------------------
# 12. Metadata, summary, and ZIP export.
# -----------------------------------------------------------------------------
metadata = {
    "project": "MorphoGraph-MOF",
    "input_cif": os.path.abspath(CIF_PATH),
    "formula": formula,
    "input_atom_count": int(len(input_structure)),
    "working_cell_atom_count": int(len(working_structure)),
    "working_lattice_matrix_A": np.asarray(working_structure.lattice.matrix).tolist(),
    "standardization_notes": standardization_notes,
    "parameters": {
        "max_miller_index": MAX_MILLER_INDEX,
        "slab_thickness_A": SLAB_THICKNESS_A,
        "vacuum_A": VACUUM_A,
        "force_tolerance_eV_A": FORCE_TOL_EV_A,
        "max_relax_steps": MAX_RELAX_STEPS,
        "particle_display_radius_A_visualization_only": PARTICLE_RADIUS_A,
        "bulk_view_repeats_per_axis": DISPLAY_REPEATS,
        "symmetry_tolerance_A": SYMPREC_A,
        "surface_fingerprint_depth_A": SURFACE_FINGERPRINT_DEPTH_A,
        "fingerprint_distance_quantization_A": FINGERPRINT_DISTANCE_TOL_A,
        "pair_fit_rms_tolerance_eV_A2": PAIR_FIT_RMS_TOL_EV_A2,
        "termination_shift_deduplication_tolerance_fractional": TERMINATION_SHIFT_TOL_FRACTIONAL,
    },
    "bulk_reference": {
        "available": bool(model is not None), "converged": bool(bulk_ok),
        "energy_total_eV": bulk_total_energy_eV, "energy_per_atom_eV": bulk_energy_per_atom_eV,
        "max_force_eV_A": bulk_fmax_eV_A, "error": bulk_error,
    },
    "miller_representatives": [list(x) for x in miller_reps],
    "directed_miller_orbits": {str(k): [list(x) for x in v] for k, v in miller_orbits.items()},
    "miller_generation_notes": miller_notes,
    "surface_pair_record_count": int(len(records_df)),
    "surface_pair_status_counts": {str(k): int(v) for k, v in records_df["status"].value_counts().items()} if len(records_df) else {},
    "surface_fit_summary": fit_summary_df.to_dict(orient="records"),
    "individual_surface_energy_count": int(len(surface_gamma_df)),
    "valid_signed_direction_energy_count": int(len(direction_gamma_J_m2)),
    "direction_energy_notes": direction_gamma_notes,
    "morphologies": morphology_rows,
    "connectivity_note": connectivity_note,
    "methodology": [
        "Stoichiometric slabs only are used for the simple bulk-reference pair-energy equations.",
        "A relaxed slab verified symmetric by pymatgen is constrained to one shared top/bottom surface-state unknown in the pair-energy equations.",
        "For asymmetric slabs, (E_slab - N*E_bulk)/A is treated as gamma_top + gamma_bottom, not as one surface energy.",
        "Slab generation tries the primitive surface cell first and retries with a non-primitive surface cell if slab generation raises an exception; the chosen mode is recorded per facet.",
        "Individual termination energies are returned only when the surface-pair linear system, fingerprinted from relaxed surface states, is full rank; if redundant equations exist, their RMS residual must meet the explicit user tolerance.",
        "A full-rank square pair system can identify energies algebraically but has no independent redundancy-based residual validation; this is marked in the fit CSV and requires external convergence/validation.",
        "A CHGNet Wulff shape is created only when every requested directed facet class has valid positive surface-energy estimates.",
        "No missing MLIP surface energies are filled with equal, mirrored, random, or guessed values.",
        "BFDH-like and equal-gamma shapes are geometry controls, not MLIP surface-energy predictions.",
        "Atomistic particles are geometric cuts of the periodic bulk; they are not passivated or separately relaxed nanoparticles.",
        "CrystalNN connectivity is inferred for display and must be chemically checked for unusual metal coordination or disordered structures.",
        "CHGNet is not automatically validated for a given MOF; validate representative surface energies against DFT-D3 and perform slab-thickness/vacuum convergence studies.",
        "This code creates training labels/baselines; it does not train the proposed cross-MOF GNN.",
    ],
}
metadata_path = RESULTS_DIR / f"{safe_name}_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, default=str)

print("\n================ FINAL SUMMARY ================")
print(f"Bulk reference converged: {bulk_ok}")
print(f"Pair-energy records     : {len(records_df)}")
print(f"Individual termination gammas: {len(surface_gamma_df)}")
print(f"Signed direction gammas : {len(direction_gamma_J_m2)} / {len(all_directed_hkls)} directed primitive indices")
print("Surface-pair statuses:")
if len(records_df):
    print(records_df["status"].value_counts(dropna=False).to_string())
print("Morphology outputs:")
if len(morphology_df):
    display(morphology_df)
print("\nOutput files:")
for path in (records_csv, fit_csv, surface_gamma_csv, morphology_csv, metadata_path):
    print(path.resolve())

archive_path = Path(f"{safe_name}_MorphoGraph_MOF_revised_results_{RUN_ID}.zip")
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as zf:
    for path in RESULTS_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(Path(RESULTS_DIR.name) / path.relative_to(RESULTS_DIR)))
print(f"\nZIP archive created: {archive_path.resolve()}")
try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    print("Outside Colab: retrieve the ZIP from the working directory.")
